In [ ]:
!pip install scikit-surprise

In [ ]:
!pip install scikit-surprise numpy==1.26.4

In [ ]:
from surprise import Dataset, SVD
from surprise.model_selection import train_test_split

data = Dataset.load_builtin('ml-100k')
train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


In [ ]:
model = SVD()
model.fit(train_set)
predictions = model.test(test_set)

In [ ]:
from surprise import accuracy

accuracy.rmse(predictions)

RMSE: 0.9362


0.936244416477391

In [ ]:
from surprise.model_selection import cross_validate

cross_validate(model, data, measures=["RMSE", "MAE"], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9401  0.9293  0.9411  0.9351  0.9340  0.9359  0.0043  
MAE (testset)     0.7406  0.7325  0.7408  0.7375  0.7381  0.7379  0.0030  
Fit time          1.41    1.68    1.20    1.20    1.21    1.34    0.19    
Test time         0.17    0.19    0.10    0.19    0.10    0.15    0.04    


{'test_rmse': array([0.94007483, 0.92932649, 0.94106688, 0.93513993, 0.93400913]),
 'test_mae': array([0.74056705, 0.73253201, 0.74079787, 0.73748572, 0.7381252 ]),
 'fit_time': (1.4091124534606934,
  1.6768076419830322,
  1.201892614364624,
  1.1963958740234375,
  1.2071201801300049),
 'test_time': (0.17232513427734375,
  0.1854245662689209,
  0.09654641151428223,
  0.1887049674987793,
  0.09762001037597656)}

In [ ]:
from surprise.model_selection import GridSearchCV

param_grid = {
    "n_epochs": [5, 10],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.4, 0.6]
}

gs = GridSearchCV(SVD, param_grid, measures=["RMSE", "MAE"], cv=3)
gs.fit(data)

print(gs.best_score["rmse"])
print(gs.best_params["rmse"])
print(gs.best_score["mae"])
print(gs.best_params["mae"])

0.9629254109126221
{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.4}
0.7715332993690085
{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.4}


In [ ]:
from surprise import SVDpp, NMF
import numpy as np

for name, algo in [('SVD', SVD()), ('SVD++', SVDpp()), ('NMF', NMF())]:
  results = cross_validate(algo, data, measures=["RMSE", "MAE"], cv=5, verbose=True)
  print(f"{name} RMSE: {np.mean(results['test_rmse'])}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9367  0.9270  0.9344  0.9387  0.9411  0.9356  0.0048  
MAE (testset)     0.7396  0.7281  0.7379  0.7377  0.7430  0.7373  0.0050  
Fit time          1.69    1.23    1.25    1.22    1.25    1.33    0.18    
Test time         0.10    0.26    0.10    0.10    0.11    0.13    0.06    
0.9355746915868368
Evaluating RMSE, MAE of algorithm SVDpp on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9202  0.9238  0.9222  0.9166  0.9150  0.9195  0.0033  
MAE (testset)     0.7203  0.7244  0.7248  0.7219  0.7156  0.7214  0.0033  
Fit time          26.03   29.25   25.32   25.70   24.71   26.20   1.59    
Test time         6.09    3.98    5.84    5.35    4.56    5.16    0.79    
0.91954825060511
Evaluating RMSE, MAE of algorithm NMF on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  

## Висновок:

Алгоритм SVD++ має нижчі значення середньоквадратичної та абсолютної середньої похибки у порівнні з звичайним SVD та NMF. Тим не менш, навчання моделі SVD++ проходить довше через те, що вона використовує більше даних. SVD використовує лише актуальні оцінки, які проставив юзер. Навіть якщо юзер подивився 50 фільмів, а виставив 10 оцінок, то SVD та NMF буде враховувати лише їх. NMF у свою чергу показав найгірший результат і у своїй суті він використовує лише невідʼємні числа. Тобто, наприклад, якщо ми оцінюємо фільм питачи себе "наскільки він комедія", то тут NMF може мінімально нам сказати лише 0, що буде означати, що це не комедія, у свою чергу обидва SVD та SVD++ використовують значення з мінусом, що можна інтерпретувати як характеристику "анти-комедія", якщо число відʼємне.
І на відмінність від SVD, SVD++ вже враховує не тільки оцінки, а й сам факт того, що юзер подивився фільм.